# PyTorch Primer

## Question

Why do a `.view()` and then `.transpose()`? Why not just do `.view()` here?

```python
q = q.view(batch_size, seq_len, self.n_head, head_size).transpose(1, 2) 
```

## Answer
It’s a great question—it feels like you should be able to just "shape it correctly" from the start. However, the reason you can’t simply view it into that final shape comes down to how data is laid out in your computer's memory.

1. Memory is a flat line

Even though we think of tensors as multi-dimensional cubes, the computer stores them as one long, continuous row of numbers.

When you create the Query tensor via a Linear layer, the data is generated in this order:
[Batch, Sequence, Embedding]

Inside that Embedding chunk, the values for Head 1, Head 2, and Head 3 are sitting right next to each other for Word A, then the heads for Word B, and so on.

2. The "Jumble" Problem

If you try to view(batch_size, n_head, seq_len, head_size) directly, you are telling the computer: "Take the first chunk of memory and call it Head 1's entire sequence."

But that first chunk of memory actually contains:

* Word A (Head 1)
* Word A (Head 2)
* Word A (Head 3)

By forcing a view without a transpose, you accidentally jumble the words. You'd end up with a "Head 1" that is actually a weird salad of the first few dimensions of every single word in the sequence.
The "Packing" Analogy

Imagine you have 2 boxes (Heads), and each box needs 2 items (Words).

* The Linear Layer outputs them like this: [Word1_H1, Word1_H2, Word2_H1, Word2_H2]
* What you want for Attention: * Head 1: [Word1, Word2]
    - Head 2: [Word1, Word2]

If you just use view:
The computer takes the first two items it sees and puts them in Head 1.
* Head 1 gets: [Word1_H1, Word1_H2] (It's missing Word 2 entirely!)

By using view + transpose:

* View separates the heads but keeps them at the word level: [[Word1_H1, Word1_H2], [Word2_H1, Word2_H2]]
* Transpose physically moves the memory (or the pointer to it) so that all "Head 1" values are grouped together: [[Word1_H1, Word2_H1], [Word1_H2, Word2_H2]]

In [3]:
import torch

# Create dummy data: 1 batch, 2 words, 4 embedding size (2 heads x 2 size)
x = torch.arange(8).view(1, 2, 4)

print(x)

# This is the correct way
correct = x.view(1, 2, 2, 2).transpose(1, 2)

# This is what happens if you skip transpose
wrong = x.view(1, 2, 2, 2)

print("\nCorrect Head 0:\n", correct[0, 0])  # Shows data for Word 1 and Word 2
print("\nWrong Head 0:\n", wrong[0, 0])  # Only shows data from Word 1

tensor([[[0, 1, 2, 3],
         [4, 5, 6, 7]]])

Correct Head 0:
 tensor([[0, 1],
        [4, 5]])

Wrong Head 0:
 tensor([[0, 1],
        [2, 3]])


## nn.Embedding

In [1]:
import torch
import torch.nn as nn

/Users/jae.r/projects/ml-study/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [22]:
n_embeddings = 256
vocab_size = 50_000
embedding = nn.Embedding(vocab_size, n_embeddings)
embedding

Embedding(50000, 256)

In [ ]:
tensor = torch.tensor([1, 2, 3, 4, 5])
embedding(tensor).size()

torch.Size([5, 256])

In [ ]:
# output is nonsensical
embedding(tensor)

tensor([[ 1.8914, -0.4755, -0.0843,  ..., -1.3005,  1.9308,  0.5283],
        [ 0.6178,  1.6125, -0.0932,  ..., -1.5397, -1.1064,  0.5807],
        [-0.3639,  0.4687,  0.3182,  ...,  0.5042,  0.6992, -0.4298],
        [ 1.6289, -0.7672,  0.5486,  ..., -0.0578,  0.7785, -0.9283],
        [ 1.3249,  0.1645,  1.0010,  ...,  0.1643, -0.6110, -0.2282]],
       grad_fn=<EmbeddingBackward0>)